# <center>  Python for biologists

## Lecture 5, Modules
30.09.2026

***Микрокоммент***. 

Конспект лекции приведен в jupyter notebook. У jupyter есть некоторые особенности при работе с модулями. Мы еще с этим познакомимся чуть позже, но пока несколько вещей которые тут будут использоваться.

(Если вы работаете просто в IDE и запускаете скрипт, то для вас все как обычно)

1) **Магические ячейки**

    В Jupyter можно ставить знаки `%` `%%` чтобы модифицировать поведение ячейки. У нас будет про это подробнее, но пока просто знайте, что:
    ```python
    %%file script.py
    
    ``` 
    написанное в начале ячейки - создает файл *script.py* и записывает туда содержимое ячейки.
    
2) **Переимпортирование**

    По-умолчанию Jupyter не импортирует заново ранее импортированные модули (даже если запустить код импортирования). То есть обновления в них не подтянутся. Можно либо перезапустить ядро (Kernel restart), либо выполнить следующий код:

In [2]:
%load_ext autoreload

In [4]:
%autoreload 2 

*2 это код означающий "включи переимпортирование"*

### <center> Поехали!

---

# Modules

- Модуль это просто скрипт
- В момент import'а весь модуль исполняется как обычный код
- Его переменные и другие объекты доступны через префикс - имя модуля
- Есть специальные правила где искать эти самые модули (`sys.path`)

Зачем нужны модули? Чтобы использовать уже написанный код - свой или чужой - и не переписывать его. Например, функции из домашки можно импортировать в любую другую программу.

Терминология (на практике слова часто взаимозаменяемы):
- **Модуль** - один `.py` файл
- **Пакет** (package) - набор нескольких `.py` файлов, по сути папка с модулями
- **Библиотека** - большой развитый пакет (или набор пакетов)

Вместе с питоном сразу идет **стандартная библиотека** (stdlib) - много готовых модулей: `os` и `sys` для работы с операционной системой, `time` и `datetime` для времени, `math` для математики и т.д. Ничего устанавливать не нужно, достаточно импортировать. Список самых полезных - в конце конспекта.

In [1]:
import math

print(math.pi)
print(math.sin(30))

3.141592653589793
-0.9880316240928618


Обращение к объектам модуля - через точку: `имя_модуля.объект`.

Теперь сделаем свой модуль. Модуль - это просто `.py` файл, никаких дополнительных настроек не нужно. Кладем его рядом со скриптом (в ту же папку) и импортируем по имени файла без `.py`.

In [2]:
%%file my_module.py 
print('Hi, i am your module!')

pi = 3.14


def sin(x):
    return x

Overwriting my_module.py


В момент импорта идёт печать - это потому что скрипт модуля просто исполняется

In [3]:
import my_module

Hi, i am your module!


Что происходит при `import my_module`:
1. Питон исполняет весь файл `my_module.py` построчно, как обычный скрипт (как `python my_module.py`)
2. Запоминает все созданные объекты (переменные, функции, ...) и дает к ним доступ через имя модуля

Поэтому в модуле на "верхнем уровне" должны быть только определения - никаких `print`, вызовов функций и тем более `input()`. Например, импорт такого модуля просто зависнет в ожидании ввода:

```python
def sin(x):
    return x

x = float(input())
print(sin(x))
```

Как сделать правильно - ниже, в `if __name__ == '__main__'`.

Обращение идёт как обычно через точку:

In [4]:
print(my_module.pi)

3.14


Это обычная переменная, ее можно перезаписать

In [5]:
my_module.pi = 4
print(my_module.pi)

4


Переменные модуля живут в своем пространстве имен и не пересекаются с нашими:

In [ ]:
pi = 3.14159

print(pi)  # наша переменная
print(my_module.pi)  # переменная модуля

А если положить модуль в какую-то папку?

In [ ]:
! mkdir scripts

In [7]:
%%file scripts/my_math.py 

pi = 3.1415


Overwriting scripts/my_math.py


In [8]:
import scripts.my_math

In [9]:
! cat scripts/my_math.py


pi = 3.1415


In [10]:
scripts.my_math.pi

3.1415

Промежуточные папки перечисляются через точку, вложенность может быть любой - по сути это уже пакет.

## Как еще по другому можно импортировать?

- Создавая alias - новое имя для модуля (`import ... as ...`)
- Импортируя только часть функций (при этом код все равно выполняется целиком, просто ненужные функции удаляются, `from ... import ...`)

*Перед каждым примером ниже перезапускаем ядро (Kernel → Restart), иначе модуль уже импортирован и заново не исполнится*

In [1]:
import my_module as mm

Hi, i am your module!


In [2]:
print(my_module.pi)

NameError: name 'my_module' is not defined

In [3]:
print(mm.pi)

3.14


После `as` модуль доступен только по новому имени.

Есть общепринятые сокращения: `import numpy as np`, `import pandas as pd`. Свои лучше не придумывать без нужды - `math.pi` читается понятнее, чем `mt.pi`.

In [1]:
from my_module import sin as my_sin

Hi, i am your module!


In [2]:
print(my_sin(30))

30


`from ... import ...` кладет объект прямо к нам, без префикса. Если такое имя уже было - оно молча перезапишется (как обычное `pi = ...`):

In [ ]:
pi = 3
from math import pi

print(pi)

Поэтому в ~90% случаев лучше обычный `import module` - так коллизий точно не будет. `from ... import ...` - когда нужен конкретный объект и вы уверены, что ничего не перезапишете.

In [1]:
from my_module import *

Hi, i am my_module!


In [3]:
print(sin(30))

0


In [4]:
print(pi)

3


Иногда люди импортируют все функции таким образом:

```
from ... import *
```

Так делать нельзя, это плохо по двум причинам:
- Опасно для работы кода, могут быть коллизии и перезаписывания каких-то функций и переменных
- Это очень плохо читается, не знаешь откуда тот или иной объект взялся

Пример: в `math` есть свой `pow`, и он молча заменит встроенный:

In [ ]:
print(pow(2, 10))  # встроенный pow

from math import *

print(pow(2, 10))  # уже math.pow, возвращает float

## Где питон ищет модули

Модуль - это файл, и у него есть путь. Например, модуль `this`:

In [ ]:
import this

In [ ]:
this.__file__

In [ ]:
! cat {this.__file__}

Внутри тот же текст, но зашифрованный шифром Цезаря (ROT13). Шутка разработчиков: дзен питона написан с нарушением всех правил дзена питона)

Список папок, где питон ищет модули, лежит в `sys.path`:

In [ ]:
import sys

print(*sys.path, sep='\n')

Питон идет по списку сверху вниз:
- Первой идет папка запускаемого скрипта - поэтому модуль рядом и находится
- Потом стандартная библиотека (`.../lib/python3.X`). В `lib-dynload` - модули, написанные на C (например, `math`)
- `site-packages` - сюда ставится всё, что вы устанавливаете через `pip`/`conda`

Чтобы питон нашел модуль в другом месте, добавьте путь в `sys.path` - это обычный список:

In [ ]:
sys.path.append('/path/to/my/modules')  # можно и относительный путь

Это работает только пока жив интерпретатор, поэтому `sys.path.append` пишут в начале скрипта.

Альтернатива - переменная окружения `PYTHONPATH` (аналог `PATH` в bash). Чтобы она задавалась всегда, пропишите в `~/.bashrc` (или `~/.zshrc`):
```bash
export PYTHONPATH=$PYTHONPATH:/path/to/my/modules
```

## `__name__` и `if __name__ == '__main__'`

У каждого модуля есть служебная переменная `__name__`. У запускаемого скрипта (и у ноутбука) это `'__main__'`:

In [ ]:
__name__

Добавим в модуль `print(__name__)` (и снова перезапустим ядро):

In [ ]:
%%file my_module.py
print(__name__)

pi = 3.14


def sin(x):
    return x

In [1]:
import my_module

my_module


In [10]:
my_module.__name__

'my_module'

In [2]:
! python my_module.py

__main__


При импорте `__name__` - это имя модуля, при запуске напрямую - `'__main__'`. У модуля из папки - полный путь через точку:

In [3]:
import scripts.my_math

In [4]:
scripts.my_math.__name__

'scripts.my_math'

На этом строится конструкция `if __name__ == '__main__':` - код внутри выполняется только при запуске файла как скрипта, а при импорте нет.

Так один файл может быть и модулем (импортируем функции), и самостоятельным инструментом для терминала. И `input()` при импорте больше ничего не вешает:

In [ ]:
%%file my_module.py
pi = 3.14


def sin(x):
    return x


if __name__ == '__main__':
    x = float(input('Enter your number: '))
    print(sin(x))

In [1]:
import my_module

In [ ]:
my_module.sin(0.5)  # импорт прошел, input не сработал

In [ ]:
! echo 0.5 | python my_module.py  # в терминале: python my_module.py и ввести число

## Документация модулей

`help` работает и для модулей - покажет описание, функции, переменные и файл:

In [ ]:
import math

help(math)

Описание своего модуля - докстринг в самом начале файла (тройные двойные кавычки). Докстринги функций и аннотации типов (было в лекции про функции) тоже попадут в `help`. После изменения файла не забываем перезапустить ядро:

In [ ]:
%%file my_module.py
"""
Very approximate math module.

Works for small x only!
"""

pi: float = 3.14


def sin(x: float) -> float:
    """
    Approximate sine for small x

    Arguments:
    - x (float): angle in radians

    Return:
    - float, approximate sine
    """
    return x


if __name__ == '__main__':
    x = float(input('Enter your number: '))
    print(sin(x))

In [ ]:
import my_module

help(my_module)

In [ ]:
help(my_module.sin)

- Питон докстринги и аннотации игнорирует - это подсказка для людей (и для IDE и тайп-чекеров вроде `mypy`)
- Аннотации переменных (`pi: float = 3.14`) пишут редко, аннотации функций - почти всегда

**В домашках теперь:** аннотации типов - у всех функций, докстринги - у основных (у мелких вспомогательных можно без них)

## Standard library modules

Here are most used modules in the python standard library:

- sys \*\*
- os \*\*\*
- re \*\*
- os.path \*\*\*, glob, pathlib
- argparse \*
- venv \*
- datetime \*
- itertools, functools \*
- multiprocessing, threading \*
- collections
- copy
- pprint
- typing
- gc

**sys**:
- exit
- getsizeof
- path
- stdin, stdout, stderr \*\*\*

**os**:
- chdir, getcwd
- mkdir
- chmod, chown
- listdir \*\*
- rename
- cpu_count

**os.path**:
- basename
- exists
- isfile, isdir \*
- join \*\*\*
